# Tiny World Models: JEPA-lite Starting Point

This notebook reconstructs the foundation we already built: a bouncing-ball world, pixel observations, temporal training examples, and a JEPA-like model skeleton.

**Stopping point:** the online encoder, target encoder, and predictor are wired for a forward pass. We will build the training loop and evaluation experiments together from there.

## Goal

Learn a latent state that captures the predictable structure of a small physical world. Later we will test whether that state contains position and velocity, supports long rollouts, and can be used for planning.

## Setup

The notebook automatically uses a CUDA GPU when one is available and otherwise runs on CPU.

In [ ]:
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Steps

### 1. Define the hidden physical world

The simulator has access to the true state: position, velocity, and radius. The learning model will receive only rendered pixels.

In [ ]:
DT = 0.05
RADIUS = 0.05

def step(state, dt=DT):
    pos = state["pos"].copy()
    vel = state["vel"].copy()
    radius = state["radius"]

    pos = pos + vel * dt

    for axis in range(2):
        if pos[axis] < radius:
            pos[axis] = radius
            vel[axis] *= -1
        elif pos[axis] > 1.0 - radius:
            pos[axis] = 1.0 - radius
            vel[axis] *= -1

    return {"pos": pos, "vel": vel, "radius": radius}

def random_state(rng):
    return {
        "pos": rng.uniform(RADIUS, 1.0 - RADIUS, size=2).astype(np.float32),
        "vel": rng.uniform(-0.5, 0.5, size=2).astype(np.float32),
        "radius": RADIUS,
    }

def rollout(initial_state, steps):
    trajectory = [initial_state]
    for _ in range(steps):
        trajectory.append(step(trajectory[-1]))
    return trajectory

In [ ]:
example_trajectory = rollout(random_state(rng), steps=40)
positions = np.stack([state["pos"] for state in example_trajectory])

positions.shape, positions[:3]

### 2. Render hidden state to pixels

Rendering deliberately removes explicit velocity. The model must infer motion from observations across time.

In [ ]:
IMAGE_SIZE = 64

def render_ball(state, size=IMAGE_SIZE):
    pos = state["pos"]
    radius = state["radius"]

    ys, xs = np.mgrid[0:size, 0:size]
    x_world = (xs + 0.5) / size
    y_world = (ys + 0.5) / size
    distance_squared = (x_world - pos[0]) ** 2 + (y_world - pos[1]) ** 2

    image = np.zeros((size, size), dtype=np.float32)
    image[distance_squared <= radius ** 2] = 1.0
    return image

In [ ]:
example_frames = np.stack([render_ball(state) for state in example_trajectory])
frame_indices = [0, 8, 16, 24, 32, 40]

fig, axes = plt.subplots(1, len(frame_indices), figsize=(12, 2))
for axis, frame_index in zip(axes, frame_indices):
    axis.imshow(example_frames[frame_index], cmap="gray", vmin=0, vmax=1)
    axis.set_title(f"t={frame_index}")
    axis.axis("off")
plt.tight_layout()

### 3. Build temporal examples

Each example has two context frames and one future target frame:

`(frame_t, frame_t+1) -> frame_t+2`

We split complete trajectories before extracting examples, so frames from one trajectory cannot appear in both training and validation data.

In [ ]:
NUM_TRAJECTORIES = 64
STEPS_PER_TRAJECTORY = 40
TRAIN_FRACTION = 0.8

trajectories = [
    rollout(random_state(rng), steps=STEPS_PER_TRAJECTORY)
    for _ in range(NUM_TRAJECTORIES)
]

videos = np.stack([
    np.stack([render_ball(state) for state in trajectory])
    for trajectory in trajectories
])

split_index = int(NUM_TRAJECTORIES * TRAIN_FRACTION)
train_videos = videos[:split_index]
validation_videos = videos[split_index:]

videos.shape, train_videos.shape, validation_videos.shape

In [ ]:
def make_examples(video_batch):
    context = np.stack(
        [video_batch[:, :-2], video_batch[:, 1:-1]],
        axis=2,
    )
    target = video_batch[:, 2:, None, :, :]

    context = context.reshape(-1, 2, IMAGE_SIZE, IMAGE_SIZE)
    target = target.reshape(-1, 1, IMAGE_SIZE, IMAGE_SIZE)
    return context, target

train_context, train_target = make_examples(train_videos)
validation_context, validation_target = make_examples(validation_videos)

train_context.shape, train_target.shape

### 4. Wrap examples in data loaders

A loader shuffles examples and yields small batches to the model.

In [ ]:
BATCH_SIZE = 32

train_dataset = TensorDataset(
    torch.from_numpy(train_context),
    torch.from_numpy(train_target),
)
validation_dataset = TensorDataset(
    torch.from_numpy(validation_context),
    torch.from_numpy(validation_target),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=torch.cuda.is_available(),
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
)

batch_context, batch_target = next(iter(train_loader))
batch_context.shape, batch_target.shape

### 5. Define the encoder and latent predictor

The encoder maps one `1 x 64 x 64` frame to one 64-number latent vector. The predictor uses two consecutive latent vectors to predict the future latent vector.

In [ ]:
LATENT_DIM = 64

class FrameEncoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, latent_dim),
        )

    def forward(self, frame):
        return self.net(frame)


class JEPAPredictor(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 * latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim),
        )

    def forward(self, z0, z1):
        return self.net(torch.cat([z0, z1], dim=-1))

### 6. Create online and EMA target encoders

The online encoder receives gradients. The target encoder starts as an independent copy, receives no gradients, and will move slowly toward the online encoder using an exponential moving average.

In [ ]:
online_encoder = FrameEncoder().to(device)
target_encoder = deepcopy(online_encoder).to(device)
target_encoder.requires_grad_(False)
target_encoder.eval()

predictor = JEPAPredictor().to(device)

optimizer = torch.optim.Adam(
    list(online_encoder.parameters()) + list(predictor.parameters()),
    lr=1e-3,
)

In [ ]:
@torch.no_grad()
def update_target_encoder(online_encoder, target_encoder, momentum=0.99):
    for online_param, target_param in zip(
        online_encoder.parameters(),
        target_encoder.parameters(),
    ):
        target_param.mul_(momentum)
        target_param.add_(online_param, alpha=1.0 - momentum)

## Checks

The encoders should begin with equal parameter values but must be separate objects. The target encoder should have no trainable parameters.

In [ ]:
same_values = all(
    torch.equal(online_param, target_param)
    for online_param, target_param in zip(
        online_encoder.parameters(),
        target_encoder.parameters(),
    )
)
same_objects = all(
    online_param is target_param
    for online_param, target_param in zip(
        online_encoder.parameters(),
        target_encoder.parameters(),
    )
)
target_trainable_parameters = sum(
    parameter.numel()
    for parameter in target_encoder.parameters()
    if parameter.requires_grad
)

same_values, same_objects, target_trainable_parameters

A final forward pass verifies the tensor wiring without training anything yet.

In [ ]:
batch_context = batch_context.to(device)
batch_target = batch_target.to(device)

frame_0 = batch_context[:, 0:1]
frame_1 = batch_context[:, 1:2]
frame_2 = batch_target

with torch.no_grad():
    z0 = online_encoder(frame_0)
    z1 = online_encoder(frame_1)
    predicted_z2 = predictor(z0, z1)
    target_z2 = target_encoder(frame_2)

z0.shape, z1.shape, predicted_z2.shape, target_z2.shape

## Next Steps

We will implement these one at a time:

1. Write one JEPA training step and understand exactly where gradients flow.
2. Build training and validation loops; monitor prediction loss and latent variance.
3. Freeze the encoder and probe position and velocity.
4. Evaluate 1-, 5-, 10-, and 20-step latent rollouts.
5. Add actions and use the learned model for model predictive control.

Do not continue by filling all of these in at once. The next learning step is the single JEPA training step.